# Practice 5-3. 그래프 RAG 질의 — 로컬 검색 vs 글로벌 검색

`02-graphrag-knowledge-graph-indexing.ipynb`에서 구축한 그래프 DB(`working_directory`)를 대상으로, 로컬 검색(local search)과
글로벌 검색(global search)의 차이를 실습으로 비교합니다.

- **로컬 검색**: 질문과 관련 있는 엔티티를 먼저 찾고, 그 엔티티에 연결된 텍스트 청크·관계·커뮤니티 리포트를
  모아 답변을 생성합니다. 특정 인물·조직 등 명시적인 질문에 적합합니다.
- **글로벌 검색**: 커뮤니티 리포트 전체를 맵-리듀스로 훑어 답변을 생성합니다. 문서 전체를 아우르는 포괄적인
  질문에 적합합니다.

책과 동일하게 **포괄적인 질문**과 **세부적인 질문**을 각각 던져, 두 검색 방식의 답변 범위와 깊이가 어떻게
달라지는지 비교합니다. 이 노트북의 대상 문서는 조지 게어 헨리(George Garr Henry)의 『How to Invest Money』
(1908)입니다.

## 0. 환경 설정

In [1]:
from pathlib import Path

WORKING_DIR = Path("working_directory")
assert (WORKING_DIR / "output" / "community_reports.parquet").exists(), \
    "practice5-2를 먼저 실행해 그래프 DB를 구축하세요"
print("작업 디렉터리:", WORKING_DIR.resolve())

작업 디렉터리: /workspace/day_05/working_directory


## 1. 포괄적인 질문 비교

**포괄적인 질문**: 이 책 전반에서 다루는 투자 대상(증권)의 종류와 저자가 강조하는 투자 원칙은 무엇인가요?

이 질문은 책 전체에 걸친 내용(철도채권, 부동산 담보대출, 산업채권, 공공시설채권, 지방채, 주식 등 다양한
투자 수단과 원칙)을 종합적으로 파악해야 정확한 답변이 가능합니다. 먼저 글로벌 검색을 수행합니다.

In [2]:
!graphrag query --root ./working_directory --method global "이 책 전반에서 다루는 투자 대상(증권)의 종류와 저자가 강조하는 투자 원칙은 무엇인가요?"

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
## 투자 대상 및 핵심 투자 원칙 요약

제공된 분석 보고서들을 종합해 볼 때, 투자 대상(증권)의 종류와 투자 시 고려해야 할 핵심 원칙에 대해 다음과 같이 정리할 수 있습니다.

### 1. 투자 대상(증권)의 종류

투자 대상의 종류에 관해서는 Railroad Finance and Securities 커뮤니티를 중심으로 Equipment Bonds, Mortgage Bonds, Corporate Structures (Railroad Company, Trustee Company), 그리고 EQUITIES 등이 핵심 구조를 이룬다고 분석됩니다 [Data: Reports (2, 9)].

### 2. 강조되는 투자 원칙 및 품질 기준

투자 원칙 측면에서, 증권을 선택할 때 INVESTORS는 다섯 가지 핵심 품질을 기준으로 삼아야 한다고 강조합니다. 이 다섯 가지 핵심 품질은 안전성(safety), 소득률(income rate), 현금화 가능성(liquidity), 가치 상승 전망(appreciation outlook), 그리고 시장 가격 안정성(market price stability)입니다 [Data: Reports (1, 45, 7)].

**세부적인 투자 분석 관점:**

*   **위험 분배 원칙:** Railroad Bonds에 대한 투자 분석에서는 위험 분배 원칙이 핵심적으로 다루어집니다. 이와 관련하여 Interest Requirement를 사용하여 채권의 보안 수준을 평가하며, 이자 수익이 요구 이자율의 두 배를 초과할 경우 더 높은 이자 요구 사항으로 보호될 가능성이 있다고 판단됩니다 [Data: Reports (12, 39)].
*   **현금화 가능성 및 소득:** 투자 결정 시 BUSIN

이번에는 같은 질문을 로컬 검색으로 진행하여 결과를 비교해 보겠습니다.

In [3]:
!graphrag query --root ./working_directory --method local "이 책 전반에서 다루는 투자 대상(증권)의 종류와 저자가 강조하는 투자 원칙은 무엇인가요?"

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
제공된 보고서들을 종합해 볼 때, 이 자료들은 다양한 투자자 유형과 금융 상품 간의 복잡한 관계, 그리고 성공적인 투자를 위한 핵심 원칙들을 다루고 있습니다.

### 투자 대상(증권)의 종류

투자 대상으로는 여러 종류의 금융 상품들이 언급됩니다. 주요하게 다루어지는 증권의 종류는 다음과 같습니다:

*   **Securities (증권):** 이는 철도 채권, 지방채(municipals), 주택저당증권(mortgages), 공공사업 채권(public-utility bonds) 등 투자 선택 기준이 적용되는 금융 상품 전반을 의미합니다 [Data: Entities (18); Relationships (7)].
*   **Municipal Bonds (지방채):** 안전성과 소득과 같은 투자 선택 기준에 따라 투자자들이 선택하는 금융 상품입니다 [Data: Entities (25)].
*   **Mortgages (주택저당증권):** 투자 가치로 사용되는 증권으로, 사업 흑자(business surplus)의 요구 사항을 충족시키기 위해 채권과 신탁 증서를 포함하여 정의됩니다 [Data: Entities (26); Relationships (63, 67)].
*   **Bank and Trust-Company Stocks (은행 및 신탁 회사 주식):** 부유한 투자자들이 높은 자본 증가 가능성을 위해 선호하는 주식 유형으로, 안정적인 이익 분배 관행 때문에 주식의 장부가치가 꾸준히 상승할 것이라는 특징이 있습니다 [Data: Reports (0); Relationships (12)].
*   **Stocks vs. Bonds (주식 대 채권):** 주식은 사업체의 자산과 이익에 대한 소유권을 나타내는 반면, 채권

글로벌 검색은 커뮤니티 리포트를 맵-리듀스로 종합해 문서 전체의 주요 주제(증권 종류, 투자 원칙)를 균형 있게
요약했습니다. 로컬 검색도 여러 관련 엔티티와 원문 청크를 확장해 유사한 범위의 답변을 생성했지만, 커뮤니티 단위로
종합하는 글로벌 검색이 포괄적 질문에 더 적합한 구조임을 확인했습니다.

## 2. 세부적인 질문 비교

**세부적인 질문**: 조지 게어 헨리(George Garr Henry)는 어떤 인물이며, 어느 회사에서 어떤 직책을 맡았나요?

이 질문은 문서 전체가 아니라 저자라는 특정 엔티티 하나와 그에 연결된 관계(소속 회사, 직책)에 답이 있으므로,
로컬 검색이 더 적합할 것으로 예상됩니다. 먼저 글로벌 검색을 수행합니다.

In [4]:
!graphrag query --root ./working_directory --method global "조지 게어 헨리(George Garr Henry)는 어떤 인물이며, 어느 회사에서 어떤 직책을 맡았나요?"

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
I am sorry but I am unable to answer this question given the provided data.


이번에는 로컬 검색으로 같은 질문을 수행합니다.

In [5]:
!graphrag query --root ./working_directory --method local "조지 게어 헨리(George Garr Henry)는 어떤 인물이며, 어느 회사에서 어떤 직책을 맡았나요?"

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
제공된 데이터 테이블에는 조지 게어 헨리(George Garr Henry)에 대한 정보가 포함되어 있지 않아 해당 질문에 답변해 드릴 수 없습니다.


## 3. 정리

- **포괄적인 질문**(투자 대상의 종류·원칙)에서는 글로벌·로컬 검색 모두 관련 답변을 생성했고, 커뮤니티 리포트를
  종합하는 **글로벌 검색**이 전체 주제 요약에 더 적합했습니다.
- **세부적인 질문**(특정 인물의 경력)은 원문에 답이 있지만 두 검색 모두 답하지 못했습니다. 기본 `use_lcc=True` 구성에서
  저자가 속한 작은 연결 그래프가 커뮤니티에서 제외된 것이 원인으로, 질문 유형만으로 검색 성공을 보장할 수 없음을 보여줍니다.
- 두 방식은 `01-graphrag-lm-studio-setup.ipynb`에서 설정한 LM Studio의 `gemma-4-e2b-it` 완성 모델과 Qwen3 임베딩 모델로 동작합니다.